# LSTM seq2seq with attention tuning

In [1]:
# SET THESE 

num_samples = 2000 # put 50000
data_path = './data/WRT1V.parquet'
batch_size = 32

In [2]:
# JUST COPY AND PASTE THIS

import time 
import pandas
import random
import json, argparse

# ML
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from plotly import graph_objects as go
from sklearn import preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, roc_auc_score

# pytorch
import torch.nn as nn
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
import torch.utils

# software modules
import models_regression as Models
import utils
import dataset as Dataset

dataset = pd.read_parquet(data_path)
dataset = dataset.head(num_samples)
dataset = dataset.astype(float)
dataset = dataset.drop(['Label_10', 'Label_20', 'Label_30', 'Label_50', 'Label_100', 'Day'], axis=1)
dataset['mid_price'] = (dataset['P_Ask_1'] + dataset['P_Bid_1']) / 2
display(dataset)

# Split the dataframe into train, validation and test
dataset_train, dataset_val, dataset_test = utils.DataTools.train_val_test_split(dataset)

# Standardize the data
scaler = preprocessing.StandardScaler()
scaler.fit(dataset_train)

dataset_train[dataset_train.columns] = scaler.fit_transform(dataset_train)
dataset_val[dataset_val.columns] = scaler.fit_transform(dataset_val)
dataset_test[dataset_test.columns] = scaler.fit_transform(dataset_test)

display(dataset_train)

# Split covariates and response variables
X_train, y_train = utils.DataTools.split_x_y(dataset_train, ['mid_price'])
X_val, y_val = utils.DataTools.split_x_y(dataset_val, ['mid_price'])
X_test, y_test = utils.DataTools.split_x_y(dataset_test, ['mid_price'])

# Convert to pytorch tensors
X_train, X_val, X_test = utils.DataTools.numpy_to_tensor( X_train, X_val, X_test, dtype=torch.float32 )
y_train, y_val, y_test = utils.DataTools.numpy_to_tensor( y_train, y_val, y_test, dtype=torch.float32 )

# Create Torch Dataset
lookback_period = 100
seq_len_y = 50
dataset_train = Dataset.TimeSeriesDataset(
    X         = X_train,
    y         = y_train,
    seq_len_x = lookback_period,
    seq_len_y = seq_len_y,
    offset    = lookback_period,
    channels  = False,
    task      = Dataset.TimeSeriesDataset.Task.REGRESSION
)
dataset_val = Dataset.TimeSeriesDataset(
    X         = X_val,
    y         = y_val,
    seq_len_x = lookback_period,
    seq_len_y = seq_len_y,
    offset    = lookback_period,
    channels  = False,
    task      = Dataset.TimeSeriesDataset.Task.REGRESSION
)
dataset_test = Dataset.TimeSeriesDataset(
    X         = X_test,
    y         = y_test,
    seq_len_x = lookback_period,
    seq_len_y = seq_len_y,
    offset    = lookback_period,
    channels  = False,
    task      = Dataset.TimeSeriesDataset.Task.REGRESSION
)

# Create Torch DataLoader
dataloader_train = Dataset.TimeSeriesLoader(
    dataset = dataset_train,
    batch_size = batch_size,
    shuffle = True
)
dataloader_val = Dataset.TimeSeriesLoader(
    dataset = dataset_val,
    batch_size = batch_size,
    shuffle = False
)
dataloader_test = Dataset.TimeSeriesLoader(
    dataset = dataset_test,
    batch_size = 1024,
    shuffle = False
)

#display(pd.DataFrame(dataset_train[9][0]))
#display(pd.DataFrame(dataset_train[9][1]))
#display(pd.DataFrame(dataset_train[10][0]))
#display(pd.DataFrame(dataset_train[10][1]))
#display(pd.DataFrame(dataset_train[11][0]))
#display(pd.DataFrame(dataset_train[11][1]))

print(f'Train dataset: {len(dataset_train)} samples')
print(f'Test dataset: {len(dataset_test)} samples')

dataloader_train.dataset.channels = False
dataloader_val.dataset.channels = False
dataloader_test.dataset.channels = False



,P_Ask_1,V_Ask_1,P_Bid_1,V_Bid_1,P_Ask_2,V_Ask_2,P_Bid_2,V_Bid_2,P_Ask_3,V_Ask_3,...,V_Bid_8,P_Ask_9,V_Ask_9,P_Bid_9,V_Bid_9,P_Ask_10,V_Ask_10,P_Bid_10,V_Bid_10,mid_price
0,0.962048,0.008123,0.964492,0.006633,0.962048,0.013564,0.964492,0.006650,0.961623,0.011154,...,0.019060,0.956217,0.004534,0.964476,0.004935,0.955381,0.002298,0.964048,0.025081,0.963270
1,0.961606,0.004041,0.964492,0.002583,0.961606,0.013564,0.964492,0.003317,0.961182,0.011154,...,0.009819,0.954466,0.005049,0.964920,0.019060,0.954068,0.044282,0.964936,0.004785,0.963049
2,0.961606,0.004041,0.964492,0.005917,0.961606,0.013564,0.964492,0.003317,0.961182,0.011154,...,0.009819,0.954466,0.005049,0.964920,0.019060,0.954068,0.044282,0.964936,0.004785,0.963049
3,0.961606,0.004041,0.964492,0.003117,0.961606,0.013564,0.964492,0.006650,0.961182,0.011154,...,0.009819,0.954466,0.005049,0.964920,0.019060,0.954068,0.044282,0.964936,0.004785,0.963049
4,0.961606,0.004041,0.964492,0.003117,0.961606,0.013564,0.964492,0.006650,0.961182,0.011154,...,0.009819,0.954466,0.005049,0.964920,0.019060,0.954068,0.044282,0.964936,0.004785,0.963049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,0.956311,0.005756,0.959166,0.004733,0.956311,0.152281,0.959166,0.005683,0.956330,0.024448,...,0.012136,0.950525,0.004363,0.959591,0.009996,0.950569,0.040200,0.959609,0.024804,0.957738
1996,0.956311,0.005756,0.959166,0.004733,0.956311,0.152281,0.959166,0.005683,0.956330,0.024448,...,0.012136,0.950525,0.004363,0.959591,0.009996,0.950569,0.040200,0.959609,0.024804,0.957738
1997,0.956311,0.005756,0.959166,0.004733,0.956311,0.152281,0.959166,0.005683,0.956330,0.024448,...,0.012136,0.950525,0.004363,0.959591,0.009996,0.950569,0.040200,0.959609,0.024804,0.957738
1998,0.956311,0.005756,0.959166,0.004733,0.956311,0.152281,0.959166,0.005683,0.956330,0.024448,...,0.012136,0.950525,0.004363,0.959591,0.009996,0.950569,0.040200,0.959609,0.024804,0.957738


,P_Ask_1,V_Ask_1,P_Bid_1,V_Bid_1,P_Ask_2,V_Ask_2,P_Bid_2,V_Bid_2,P_Ask_3,V_Ask_3,...,V_Bid_8,P_Ask_9,V_Ask_9,P_Bid_9,V_Bid_9,P_Ask_10,V_Ask_10,P_Bid_10,V_Bid_10,mid_price
0,0.203681,-0.395353,0.134344,0.268139,0.145807,-0.493281,0.194617,0.128313,0.097453,-0.645598,...,0.260602,0.061469,-0.602613,0.277317,-0.736715,-0.080193,-0.699857,0.252891,0.915328,0.169902
1,0.116210,-0.904593,0.134344,-0.610124,0.057857,-0.493281,0.194617,-0.616145,0.009203,-0.645598,...,-0.405392,-0.266992,-0.566333,0.361134,0.261083,-0.313443,1.573009,0.416990,-0.673677,0.126060
2,0.116210,-0.904593,0.134344,0.112726,0.057857,-0.493281,0.194617,-0.616145,0.009203,-0.645598,...,-0.405392,-0.266992,-0.566333,0.361134,0.261083,-0.313443,1.573009,0.416990,-0.673677,0.126060
3,0.116210,-0.904593,0.134344,-0.494468,0.057857,-0.493281,0.194617,0.128313,0.009203,-0.645598,...,-0.405392,-0.266992,-0.566333,0.361134,0.261083,-0.313443,1.573009,0.416990,-0.673677,0.126060
4,0.116210,-0.904593,0.134344,-0.494468,0.057857,-0.493281,0.194617,0.128313,0.009203,-0.645598,...,-0.405392,-0.266992,-0.566333,0.361134,0.261083,-0.313443,1.573009,0.416990,-0.673677,0.126060
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1395,-0.146200,0.225921,-0.302445,0.369338,-0.030091,-0.730543,-0.246120,0.128313,0.009203,0.795000,...,-0.666708,0.061469,-0.545601,-0.141772,1.321132,0.464055,-0.527160,-0.075309,-0.352327,-0.225943
1396,-0.146200,0.225921,-0.302445,0.369338,-0.118041,-0.730543,-0.246120,0.128313,0.009203,0.795000,...,1.342161,0.061469,-0.545601,-0.225590,-0.446802,0.464055,-0.527160,-0.321460,-0.870824,-0.225943
1397,-0.146200,0.735161,-0.215088,-0.812522,0.145807,1.381082,-0.157973,0.228815,0.097453,-0.627362,...,-0.666708,0.061469,-0.545601,-0.141772,1.321132,0.464055,-0.527160,-0.075309,-0.352327,-0.181848
1398,-0.146200,0.735161,-0.215088,-0.451097,0.145807,1.381082,-0.157973,-0.143415,0.097453,-0.627362,...,1.342161,0.061469,-0.545601,-0.225590,-0.446802,0.464055,-0.527160,-0.321460,-0.870824,-0.181848


Train dataset: 1200 samples
Test dataset: 100 samples


## The tuning

In [3]:
import es_tuning_LSTMSeq2SeqWithAttention_functions as esr

In [4]:
# Example usage
max_fitness_evaluations = 100        # change it to 100
num_epochs_per_nn_evaluation = 15    # change it to 30

mu_population_size = 10              # change it to 10
lambda_offsprings_size = 15          # change it to 20

init_unitsNumber_interval = [32,128]

nn_training_data = dataloader_train
nn_validation_data = dataloader_val
nn_input_size = X_train.shape[1]
nn_output_size = y_train.shape[1]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

csv_file_path = './csv_results/es_LSTMSeq2SeqWithAttention.csv'

init_layersNumber_interval = [1,1]  
init_activation_functions = [nn.ReLU] 
init_lr_interval = (5e-6, 5e-3)
init_dropout_interval = (0.0, 0.4)
maximize_fitness = False

tornamentPercentage_for_offspringGeneration = 0.2
p_crossover_vs_mutation = 0.5 # changed from 0.75 yo 0.5
structure_k_value_for_crossover = 0.3 # changed from 0 to 0.3
structure_lb_size = 32
structure_ub_size = 150
lr_k = 0
dropout_k = 0
lr_lb = 5e-6
lr_ub = 5e-3
dropout_lb = 0.0
dropout_ub = 0.4


best_individual_ever = esr.es_lambdaMu_sld_regression(
    max_fitness_evaluations = max_fitness_evaluations,
    mu_population_size = mu_population_size,
    init_layersNumber_interval = init_layersNumber_interval,
    init_unitsNumber_interval = init_unitsNumber_interval,
    init_activation_functions = init_activation_functions,
    init_lr_interval = init_lr_interval,
    init_dropout_interval = init_dropout_interval,
    nn_training_data = nn_training_data,
    nn_validation_data = nn_validation_data,
    num_epochs_per_nn_evaluation = num_epochs_per_nn_evaluation,
    nn_input_size = nn_input_size,
    nn_output_size = nn_output_size,
    device = device,
    maximize_fitness = maximize_fitness,
    lambda_offsprings_size = lambda_offsprings_size,
    tornamentPercentage_for_offspringGeneration = tornamentPercentage_for_offspringGeneration,
    p_crossover_vs_mutation = p_crossover_vs_mutation,
    structure_k_value_for_crossover = structure_k_value_for_crossover,
    structure_lb_size = structure_lb_size,
    structure_ub_size = structure_ub_size,
    lr_k = lr_k,
    dropout_k = dropout_k,
    lr_lb = lr_lb,
    lr_ub = lr_ub,
    dropout_lb = dropout_lb,
    dropout_ub = dropout_ub,
    csv_file_path=csv_file_path)

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Estimated time: 0 hours and 0 minutes
Generation 1: 2 fitness evaluations, best individual: Individual: nn_gene = [106], af = <class 'torch.nn.modules.activation.ReLU'>, lr = 0.0004438796232028312, dropout = 0.00371369374378987 | nn_sol = [106], Fitness = 0.5794345438480377


Training:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Generation 4: 11 fitness evaluations, best individual: Individual: nn_gene = [48.46538567210633], af = <class 'torch.nn.modules.activation.ReLU'>, lr = 0.0009434111894572089, dropout = 0.0630893704047395 | nn_sol = [48], Fitness = 0.5148352384567261
Best individual ever: Individual: nn_gene = [48.46538567210633], af = <class 'torch.nn.modules.activation.ReLU'>, lr = 0.0009434111894572089, dropout = 0.0630893704047395 | nn_sol = [48], Fitness = 0.5148352384567261
